# Notebook 8: Precision-Recall Curve & Average Precision (AP)

**Difficulty:** ⭐⭐⭐⭐ (4 / 5)  
**Estimated time:** 2.5 hours  
**Theme:** Credit Card Fraud Detection (1% fraud rate — severely imbalanced)

---

## Why Does This Matter?

In the previous notebook, you learned that AUC-ROC can look impressive even on severely imbalanced data — because the enormous number of true negatives keeps the False Positive Rate (FPR) artificially low.  

Credit card fraud is typically **less than 1% of all transactions**. On such data, ROC gives an optimistic picture of models that barely find any fraud. The **Precision-Recall (PR) curve** bypasses this problem entirely: it never uses True Negatives in its calculations, so the class imbalance cannot inflate the numbers.

The PR curve directly answers the two questions that matter most for fraud detection:  
- **Precision:** Of all transactions I flagged as fraud, how many actually were? (false alarm rate)  
- **Recall:** Of all actual fraud transactions, how many did I catch? (detection rate)  

---

## Real-World Analogy — The Medical Screening Test

Imagine a hospital screening tool for a rare disease affecting **1 in 100 patients**.

| Metric | Question it answers | Why it matters |
|---|---|---|
| **Recall (sensitivity)** | Of all sick patients, how many did we catch? | Missing a sick patient is dangerous |
| **Precision (PPV)** | Of all patients we flagged, how many are actually sick? | False alarms cause unnecessary treatment, anxiety, cost |

A test that says _"everyone is sick"_ has **100% recall** but only **1% precision** (99 healthy people falsely alarmed for every 1 true case).  
The PR curve plots every possible tradeoff between these two values as you vary the threshold — and **Average Precision** summarises the whole curve in one number.

In fraud detection, swap "sick patient" with "fraudulent transaction" and you have the exact same problem.

## Plain-English Concept

**The core formulas:**

$$\text{Precision} = \frac{TP}{TP + FP} \quad \text{(of all flagged, how many are real fraud?)}$$

$$\text{Recall} = \frac{TP}{TP + FN} \quad \text{(of all real fraud, how many did we catch?)}$$

Notice: **no TN anywhere**. The large majority class is invisible in both formulas.

**How the PR curve is built:**  
Sweep the threshold from high (1.0) to low (0.0):
- At high threshold: only the most suspicious transactions are flagged → high precision, low recall
- As threshold decreases: we flag more transactions → recall rises, but precision may drop as we flag lower-confidence cases
- At threshold = 0: everything flagged → recall = 1.0, precision = base rate (fraction of all transactions that are fraud)

**Baseline:**  
A random classifier that outputs scores independent of the true labels achieves precision = base rate at every recall level. On a 1% fraud dataset, the baseline is a flat horizontal line at precision = 0.01.

**Average Precision (AP):**  

$$\text{AP} = \sum_{n} (R_n - R_{n-1}) \times P_n$$

This is a step-function integral under the PR curve — a weighted mean of precisions, weighted by the increase in recall at each step.  
- AP = 1.0: perfect classifier (precision stays at 1 as recall goes from 0 to 1)
- AP = base rate: random classifier
- AP > 0.5: generally very good for fraud detection

In [ ]:
# ── Cell 3: Imports and global settings ───────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    roc_curve, roc_auc_score,
    precision_score, recall_score, f1_score,
    confusion_matrix
)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})
sns.set_palette('tab10')
SEED = 42
print('All libraries loaded successfully.')

In [ ]:
# ── Cell 4: Create severely imbalanced fraud dataset (10 000 samples, 1% fraud)
TOTAL_SAMPLES = 10_000
FRAUD_RATE    = 0.01      # 1% — 100 fraud cases out of 10 000

X, y = make_classification(
    n_samples=TOTAL_SAMPLES,
    n_features=20,
    n_informative=8,
    n_redundant=4,
    weights=[1 - FRAUD_RATE, FRAUD_RATE],
    flip_y=0.005,           # minimal label noise
    random_state=SEED
)

# Inspect class distribution
print('Dataset Overview')
print(f'  Total samples      : {len(y):,}')
print(f'  Legit transactions : {(y==0).sum():,}  ({(y==0).mean()*100:.1f}%)')
print(f'  Fraud transactions : {(y==1).sum():,}  ({(y==1).mean()*100:.1f}%)')
print(f'  Imbalance ratio    : {(y==0).sum()/(y==1).sum():.0f}:1')
print(f'\nBase rate (random classifier AP baseline): {y.mean():.4f}')

base_rate = y.mean()

In [ ]:
# ── Cell 5: Train/test split and train four classifiers ───────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)

print(f'Training set: {len(y_train):,} samples  ({y_train.sum()} fraud)')
print(f'Test set    : {len(y_test):,} samples   ({y_test.sum()} fraud)')

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Define and train all four models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=SEED),
    'k-NN (k=5)':          KNeighborsClassifier(n_neighbors=5),
    'Gaussian NB':         GaussianNB(),
    'Decision Tree':       DecisionTreeClassifier(max_depth=6, random_state=SEED)
}

fitted = {}
for name, m in models.items():
    m.fit(X_train_sc, y_train)
    fitted[name] = m
    ap = average_precision_score(y_test, m.predict_proba(X_test_sc)[:, 1])
    print(f'  Trained {name:<25} | AP = {ap:.4f}')

## Building the Precision-Recall Curve from Scratch

The algorithm mirrors the ROC curve construction but records (Precision, Recall) instead of (FPR, TPR):

1. Sort all unique prediction scores in descending order — these are our candidate thresholds.
2. For each threshold: apply it, compute the binary predictions, then compute precision and recall.
3. Collect all (recall, precision) points.

**Important edge case:** when the threshold is so high that no sample is predicted positive (`y_pred.sum() == 0`), precision is undefined. By convention, sklearn sets precision = 1 and recall = 0 at this extreme point — we follow the same convention.

In [ ]:
# ── Cell 7: PR curve from scratch ─────────────────────────────────────────
def pr_curve_scratch(y_true, y_scores):
    """
    Compute Precision-Recall curve by sweeping each unique score as a threshold.
    Returns (precisions, recalls, thresholds).
    Follows sklearn convention: precision=1, recall=0 when no positives predicted.
    """
    # Sort thresholds from highest to lowest score
    thresholds = np.sort(np.unique(y_scores))[::-1]

    precisions, recalls = [], []

    for thresh in thresholds:
        y_pred = (y_scores >= thresh).astype(int)

        # Handle edge case: nothing predicted positive
        if y_pred.sum() == 0:
            precisions.append(1.0)
            recalls.append(0.0)
            continue

        tp = int(np.sum((y_pred == 1) & (y_true == 1)))
        fp = int(np.sum((y_pred == 1) & (y_true == 0)))
        fn = int(np.sum((y_pred == 0) & (y_true == 1)))

        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0

        precisions.append(prec)
        recalls.append(rec)

    return np.array(precisions), np.array(recalls), thresholds

print('pr_curve_scratch() defined.')

In [ ]:
# ── Cell 8: Verify scratch PR curve against sklearn ────────────────────────
model_name = 'Logistic Regression'
scores_lr = fitted[model_name].predict_proba(X_test_sc)[:, 1]

# Scratch
prec_s, rec_s, thresh_s = pr_curve_scratch(y_test, scores_lr)

# sklearn (note: sklearn returns arrays with an extra point at recall=0, precision=1)
prec_sk, rec_sk, thresh_sk = precision_recall_curve(y_test, scores_lr)

# Compare on a plot
fig, ax = plt.subplots(figsize=(6.5, 5))
ax.plot(rec_sk, prec_sk, lw=3, label='sklearn', color='steelblue')
ax.step(rec_s,  prec_s,  lw=2, linestyle='--', where='post',
        label='scratch implementation', color='tomato')
ax.axhline(base_rate, color='grey', linestyle=':', lw=1.5,
           label=f'Random baseline (precision={base_rate:.3f})')
ax.set_xlabel('Recall (fraction of fraud caught)')
ax.set_ylabel('Precision (fraction of flags that are real fraud)')
ax.set_title(f'Precision-Recall Curve — {model_name}\n(scratch vs sklearn verification)')
ax.legend(fontsize=9)
ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.05)
plt.tight_layout()
plt.show()
print('Scratch curve matches sklearn — dashed line overlaps solid line.')

## Average Precision — the Step-Function Integral

Unlike AUC (which uses the trapezoidal rule), **Average Precision** uses a **step-function integral**:

$$\text{AP} = \sum_{n=1}^{N} (R_n - R_{n-1}) \times P_n$$

Each term is:
- **Width** = how much recall *increased* at this step ($R_n - R_{n-1}$)
- **Height** = precision at this step ($P_n$)

This is equivalent to computing the area under the staircase-shaped PR curve.  
The step-function approach is natural because the PR curve is inherently discrete (one point per unique score threshold).

**Interpretation:**  
AP rewards models that achieve high precision even when recall is low — i.e., the model's highest-ranked predictions are mostly real fraud. A model that scores all fraud transactions near 1.0 will have AP close to 1.

In [ ]:
# ── Cell 10: Average Precision from scratch (step-function integral) ────────
def average_precision_scratch(precisions, recalls):
    """
    Compute Average Precision as step-function integral:
      AP = sum over n of (R_n - R_{n-1}) * P_n
    Inputs should already be sorted by increasing recall.
    """
    # Sort by increasing recall
    order = np.argsort(recalls)
    prec_s = precisions[order]
    rec_s  = recalls[order]

    ap = 0.0
    for i in range(1, len(rec_s)):
        delta_recall = rec_s[i] - rec_s[i - 1]    # width of step
        ap += delta_recall * prec_s[i]             # area of rectangle

    return ap

# Verify against sklearn for all four models
print(f'{'Model':<25} {'Scratch AP':>12} {'sklearn AP':>12} {'Diff':>8}')
print('-' * 62)
for name, model in fitted.items():
    scores = model.predict_proba(X_test_sc)[:, 1]
    prec_sc, rec_sc, _ = pr_curve_scratch(y_test, scores)
    ap_sc  = average_precision_scratch(prec_sc, rec_sc)
    ap_sk  = average_precision_score(y_test, scores)
    print(f'{name:<25} {ap_sc:>12.4f} {ap_sk:>12.4f} {abs(ap_sc-ap_sk):>8.6f}')

print(f'\nRandom classifier expected AP: {base_rate:.4f}')

In [ ]:
# ── Cell 11: Plot all 4 PR curves with baseline ────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

colors = plt.cm.tab10(np.linspace(0, 0.4, 4))

for (name, model), color in zip(fitted.items(), colors):
    scores = model.predict_proba(X_test_sc)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, scores)
    ap = average_precision_score(y_test, scores)
    ax.step(rec, prec, where='post', lw=2, color=color,
            label=f'{name}  (AP = {ap:.3f})')

# Random classifier baseline — horizontal line at base_rate
ax.axhline(base_rate, color='black', linestyle='--', lw=1.5,
           label=f'Random baseline (AP = {base_rate:.3f})')

ax.set_xlabel('Recall (fraction of fraud caught)')
ax.set_ylabel('Precision (fraction of flags that are real fraud)')
ax.set_title('Precision-Recall Curves — Credit Card Fraud Detection\n'
             '(10 000 samples, 1% fraud — severe imbalance)')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.05)
plt.tight_layout()
plt.show()

## Iso-F1 Curves — Finding the Optimal Threshold

The **F1 score** is the harmonic mean of precision and recall:

$$F_1 = \frac{2 \cdot P \cdot R}{P + R}$$

For a fixed F1 value $c$, the equation becomes:
$$P = \frac{c \cdot R}{2R - c}$$

This is a **hyperbola** in (R, P) space. By overlaying multiple iso-F1 curves on the PR plot, you can visually read off the F1 score at any operating point.  

The **optimal threshold for F1** is the point on the PR curve where F1 is maximised. This is the threshold you would choose if you weight precision and recall equally.

In [ ]:
# ── Cell 13: PR curves with iso-F1 hyperbolas overlaid ────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
colors = plt.cm.tab10(np.linspace(0, 0.4, 4))

# Draw iso-F1 hyperbolas for F1 = 0.1, 0.2, 0.4, 0.6
f1_levels = [0.1, 0.2, 0.4, 0.6]
recall_grid = np.linspace(0.01, 1.0, 300)
for f1_val in f1_levels:
    # Precision from rearranged F1 formula: P = f1*R / (2R - f1)
    prec_iso = (f1_val * recall_grid) / (2 * recall_grid - f1_val)
    valid = (prec_iso > 0) & (prec_iso <= 1.0)
    ax.plot(recall_grid[valid], prec_iso[valid],
            color='lightgrey', linestyle='-', lw=0.9, zorder=1)
    # Label at the top-right of each curve
    last_valid_idx = np.where(valid)[0][-1]
    ax.annotate(f'F1={f1_val}',
                xy=(recall_grid[last_valid_idx], prec_iso[last_valid_idx]),
                fontsize=8, color='grey', va='bottom')

# Plot PR curves and mark optimal F1 threshold for each model
for (name, model), color in zip(fitted.items(), colors):
    scores = model.predict_proba(X_test_sc)[:, 1]
    prec, rec, thresh = precision_recall_curve(y_test, scores)
    ap = average_precision_score(y_test, scores)

    ax.step(rec, prec, where='post', lw=2, color=color, zorder=3,
            label=f'{name}  (AP={ap:.3f})')

    # Optimal F1 point: maximise 2PR/(P+R), avoid division by zero
    denom = prec[:-1] + rec[:-1]   # exclude last point (no threshold)
    f1_vals = np.where(denom > 0,
                       2 * prec[:-1] * rec[:-1] / denom, 0)
    best_idx = np.argmax(f1_vals)
    ax.scatter(rec[best_idx], prec[best_idx],
               s=90, color=color, marker='*', zorder=5)

# Baseline
ax.axhline(base_rate, color='black', linestyle='--', lw=1.5,
           label=f'Random baseline (AP={base_rate:.3f})')

ax.scatter([], [], s=90, marker='*', color='grey', label='Optimal F1 threshold')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('PR Curves with Iso-F1 Hyperbolas\n(★ = threshold maximising F1)')
ax.legend(loc='upper right', fontsize=8)
ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 14: Print optimal F1 thresholds for all models ───────────────────
print(f'{'Model':<25} {'Best Thresh':>12} {'Precision':>10} {'Recall':>10} {'F1':>8}')
print('-' * 70)
for name, model in fitted.items():
    scores = model.predict_proba(X_test_sc)[:, 1]
    prec, rec, thresh = precision_recall_curve(y_test, scores)

    # Best F1 (exclude the terminal point which has no threshold)
    denom = prec[:-1] + rec[:-1]
    f1_vals = np.where(denom > 0, 2 * prec[:-1] * rec[:-1] / denom, 0)
    best_idx = np.argmax(f1_vals)

    best_t  = thresh[best_idx]
    best_p  = prec[best_idx]
    best_r  = rec[best_idx]
    best_f1 = f1_vals[best_idx]
    print(f'{name:<25} {best_t:>12.4f} {best_p:>10.4f} {best_r:>10.4f} {best_f1:>8.4f}')

## ROC vs PR — A Side-by-Side Comparison

This is the core diagnostic: **on severely imbalanced data, PR curves reveal differences between models that ROC curves hide**.

The reason:
- ROC uses FPR in the x-axis → FPR denominator contains TN, which is huge for imbalanced data
- Even a mediocre model has a small FPR because TN >> FP, so all models cluster in the top-left of the ROC plot
- PR contains no TN → differences between good and mediocre models become visible

In [ ]:
# ── Cell 16: ROC vs PR side-by-side comparison ────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
colors = plt.cm.tab10(np.linspace(0, 0.4, 4))

for (name, model), color in zip(fitted.items(), colors):
    scores = model.predict_proba(X_test_sc)[:, 1]

    # ROC
    fpr, tpr, _ = roc_curve(y_test, scores)
    auc_val = roc_auc_score(y_test, scores)
    ax1.plot(fpr, tpr, lw=2, color=color,
             label=f'{name}  (AUC={auc_val:.3f})')

    # PR
    prec, rec, _ = precision_recall_curve(y_test, scores)
    ap_val = average_precision_score(y_test, scores)
    ax2.step(rec, prec, where='post', lw=2, color=color,
             label=f'{name}  (AP={ap_val:.3f})')

# ROC baseline
ax1.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Random (AUC=0.500)')
ax1.set_xlabel('False Positive Rate'); ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curves\n(1% fraud — note: all look reasonably good)')
ax1.legend(fontsize=8, loc='lower right')

# PR baseline
ax2.axhline(base_rate, color='black', linestyle='--', lw=1.2,
            label=f'Random (AP={base_rate:.3f})')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title('PR Curves\n(1% fraud — differences between models are CLEAR)')
ax2.legend(fontsize=8, loc='upper right')
ax2.set_ylim(-0.01, 1.05)

plt.suptitle('ROC vs PR — Severely Imbalanced Data (1% fraud rate)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('Key insight: ROC curves cluster together and all look similar.')
print('PR curves spread out — revealing which model truly finds fraud.')

In [ ]:
# ── Cell 17: Quantify the ROC optimism — a concrete example ───────────────
# Train an intentionally weak model: DTree with depth=1 (near-random)
weak_model = DecisionTreeClassifier(max_depth=1, random_state=SEED)
weak_model.fit(X_train_sc, y_train)
weak_scores = weak_model.predict_proba(X_test_sc)[:, 1]

# Compare weak model vs best model
best_name = 'Logistic Regression'
best_scores = fitted[best_name].predict_proba(X_test_sc)[:, 1]

roc_best = roc_auc_score(y_test, best_scores)
roc_weak = roc_auc_score(y_test, weak_scores)
ap_best  = average_precision_score(y_test, best_scores)
ap_weak  = average_precision_score(y_test, weak_scores)

print('Metric comparison — best model vs depth-1 stump (near random):')
print(f'  {'Metric':<20} {'Best model':>12} {'Weak model':>12} {'Ratio':>8}')
print('  ' + '-' * 55)
print(f'  {'AUC-ROC':<20} {roc_best:>12.4f} {roc_weak:>12.4f} {roc_best/roc_weak:>8.2f}x')
print(f'  {'Avg Precision':<20} {ap_best:>12.4f} {ap_weak:>12.4f} {ap_best/ap_weak:>8.2f}x')
print()
print('=> ROC ratio is modest; AP ratio is dramatic.')
print('   PR reveals how much better the good model really is.')

In [ ]:
# ── Cell 18: AP learning curve — how does AP improve with more training data?
# Simulate adding more training data and track AP on the fixed test set.
fractions = np.linspace(0.05, 1.0, 20)  # from 5% to 100% of training data
n_train = len(y_train)

ap_curves = {name: [] for name in fitted}
rng = np.random.default_rng(SEED)

for frac in fractions:
    n_use = max(10, int(frac * n_train))  # minimum 10 samples
    # Stratified subsample of training data
    idx_pos = np.where(y_train == 1)[0]
    idx_neg = np.where(y_train == 0)[0]
    n_pos = max(2, int(frac * len(idx_pos)))
    n_neg = min(len(idx_neg), n_use - n_pos)
    idx = np.concatenate([
        rng.choice(idx_pos, n_pos, replace=False),
        rng.choice(idx_neg, n_neg, replace=False)
    ])

    for name in fitted:
        m = models[name].__class__(**models[name].get_params())
        m.fit(X_train_sc[idx], y_train[idx])
        sc = m.predict_proba(X_test_sc)[:, 1]
        ap_curves[name].append(average_precision_score(y_test, sc))

# Plot AP learning curves
fig, ax = plt.subplots(figsize=(9, 5))
for (name, ap_vals), color in zip(ap_curves.items(),
                                   plt.cm.tab10(np.linspace(0, 0.4, 4))):
    ax.plot(fractions * 100, ap_vals, lw=2, marker='o', markersize=4,
            color=color, label=name)

ax.axhline(base_rate, color='black', linestyle='--', lw=1.2,
           label=f'Random baseline (AP={base_rate:.3f})')
ax.set_xlabel('Training data used (%)')
ax.set_ylabel('Average Precision (AP) on test set')
ax.set_title('Learning Curve for AP — How Much Data Do We Need?\n'
             '(1% fraud rate, fixed test set of 3 000 samples)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 19: Threshold selection — business-driven operating points ─────────
# In real fraud operations, the choice of threshold depends on:
# - How many analyst-hours are available to review flagged transactions (sets max FP)
# - What the cost of a missed fraud vs false alarm is

model_name = 'Logistic Regression'
scores = fitted[model_name].predict_proba(X_test_sc)[:, 1]
prec, rec, thresh = precision_recall_curve(y_test, scores)

# Scenario: operations team can review at most 50 flagged transactions per day
# Test set has 3000 samples; operational daily volume is scaled accordingly.
# Find all thresholds where the number of flagged transactions <= 50 in 3000.

print('Threshold analysis — Logistic Regression')
print(f'Total test samples: {len(y_test):,}  |  Actual fraud: {y_test.sum()}')
print()
print(f'{'Threshold':>10} {'Flagged':>9} {'Precision':>10} {'Recall':>8} {'F1':>8}')
print('-' * 52)

test_thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
for t in test_thresholds:
    y_pred_t = (scores >= t).astype(int)
    flagged   = y_pred_t.sum()
    if flagged == 0:
        print(f'{t:>10.1f} {flagged:>9}   (nothing flagged)')
        continue
    p = precision_score(y_test, y_pred_t, zero_division=0)
    r = recall_score(y_test, y_pred_t, zero_division=0)
    f = f1_score(y_test, y_pred_t, zero_division=0)
    print(f'{t:>10.1f} {flagged:>9} {p:>10.3f} {r:>8.3f} {f:>8.3f}')

In [ ]:
# ── Cell 20: Final comprehensive PR comparison dashboard ───────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
colors = plt.cm.tab10(np.linspace(0, 0.4, 4))

for idx, ((name, model), color) in enumerate(zip(fitted.items(), colors)):
    ax = axes[idx]
    scores = model.predict_proba(X_test_sc)[:, 1]
    prec, rec, thresh = precision_recall_curve(y_test, scores)
    ap_val = average_precision_score(y_test, scores)

    # Iso-F1 curves (light grey)
    recall_grid = np.linspace(0.01, 1.0, 200)
    for f1_val in [0.1, 0.2, 0.4, 0.6]:
        prec_iso = (f1_val * recall_grid) / (2 * recall_grid - f1_val)
        valid = (prec_iso > 0) & (prec_iso <= 1.0)
        ax.plot(recall_grid[valid], prec_iso[valid],
                color='lightgrey', lw=0.8, zorder=1)

    ax.step(rec, prec, where='post', lw=2, color=color, zorder=3)
    ax.fill_between(rec, prec, step='post', alpha=0.15, color=color, zorder=2)
    ax.axhline(base_rate, color='black', linestyle='--', lw=1, zorder=2)

    # Mark optimal F1
    denom = prec[:-1] + rec[:-1]
    f1_vals_all = np.where(denom > 0, 2 * prec[:-1] * rec[:-1] / denom, 0)
    best_idx = np.argmax(f1_vals_all)
    ax.scatter(rec[best_idx], prec[best_idx],
               s=120, color='black', marker='*', zorder=6,
               label=f'Best F1={f1_vals_all[best_idx]:.3f}')

    ax.set_title(f'{name}\nAP = {ap_val:.4f}  |  Baseline = {base_rate:.4f}')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.05)
    ax.legend(fontsize=8)

plt.suptitle('Precision-Recall Curves — Individual Model View\n'
             '(shaded area = AP, ★ = optimal F1 threshold, grey lines = iso-F1)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## When to Use PR Curve vs ROC Curve

| Situation | Recommended metric |
|---|---|
| Severely imbalanced data (< 5% positive class) | **PR curve / AP** |
| Balanced classes | ROC / AUC |
| FP and FN have very different costs | PR curve (or cost-sensitive analysis) |
| Comparing ranking quality of two models | AUC-ROC (clean probabilistic interpretation) |
| You care about minority class performance specifically | **PR curve / AP** |
| You want a single number for model selection | AP (imbalanced) or AUC (balanced) |

**Rule of thumb:** When your dataset has more negatives than positives by a ratio of 20:1 or more, switch to PR curves as your primary diagnostic.

---

## Summary of Formulas

| Formula | Expression | Range |
|---|---|---|
| Precision | $TP / (TP + FP)$ | 0 → 1 |
| Recall | $TP / (TP + FN)$ | 0 → 1 |
| F1 | $2PR / (P + R)$ | 0 → 1 |
| Average Precision | $\sum (R_n - R_{n-1}) \cdot P_n$ | base_rate → 1 |
| Random baseline AP | = fraction of positives (base rate) | fixed |
| ROC AUC | Trapezoidal area under ROC | 0.5 → 1 |

## Self-Check Questions

Test your understanding. Try to answer before expanding.

---

**Question 1:**  
With a 1% fraud rate, what is the expected Average Precision of a random classifier? What if the fraud rate were 50%?

<details>
<summary>Show Answer</summary>

A random classifier assigns scores that are independent of the true labels. As you lower the threshold and flag more transactions, the precision stays constant at **the base rate** (fraction of positives in the dataset), because the flagged pool mirrors the dataset distribution.

- **1% fraud rate:** Random classifier AP ≈ **0.01** (a horizontal line at precision = 0.01)
- **50% fraud rate:** Random classifier AP ≈ **0.50** (a horizontal line at precision = 0.50)

This is why you should always report AP relative to the random baseline. An AP of 0.15 on a 1% dataset is 15× above baseline — impressive. An AP of 0.15 on a 50% dataset is below baseline — terrible.

</details>

---

**Question 2:**  
Model A has AUC-ROC = 0.95 but AP = 0.12. Model B has AUC-ROC = 0.88 but AP = 0.45. For fraud detection with a 1% base rate, which model is better? Why?

<details>
<summary>Show Answer</summary>

**Model B is better** for fraud detection.

Here is why:
- With a 1% fraud rate, the 99% majority class (legit transactions) dominate the ROC curve. Model A's high AUC-ROC (0.95) may be inflated by its ability to correctly classify legit transactions, while it performs poorly on the rare fraud class.
- AP = 0.12 for Model A means: when you flag the top-ranked transactions, only 12% of the area under the PR curve is captured — barely above some random baselines, and the precision drops quickly as you try to catch more fraud.
- AP = 0.45 for Model B means: this model ranks fraud transactions far above legit ones. At many recall levels, it maintains high precision — it is a much better fraud finder.
- The PR curve never uses True Negatives, so it is not fooled by the model being good at classifying the majority class.

**Bottom line:** On imbalanced fraud data, AP is the more honest metric. Model B is the one to deploy.

</details>

---

**Question 3:**  
The PR curve generally shows precision dropping as recall increases. Why is this the general trend? Is it possible to have a PR curve that goes up AND to the right simultaneously?

<details>
<summary>Show Answer</summary>

**Why precision generally drops as recall rises:**  
As you lower the threshold to flag more transactions (increasing recall), you inevitably start flagging lower-confidence predictions. These borderline cases are more likely to be legit transactions that the model is uncertain about — so the fraction of true fraud among all flagged transactions (precision) tends to decrease.  

Formally: when you add a new transaction to the flagged set by lowering the threshold, if it is legit (very likely, since 99% are legit), precision falls. If by chance it is fraud, precision rises slightly. The law of large numbers ensures that, on average, precision decreases as recall increases.

**Can the curve go up AND to the right?**  
Yes — briefly and locally. This happens when a batch of high-confidence fraud cases all share the same predicted score. As you lower the threshold to include this cluster, you add many true positives at once (recall jumps up) while adding few or no false positives (precision stays high or even rises). This creates a step in the PR curve that goes right (recall increases) and stays flat or rises (precision does not fall). In practice, sklearn's `precision_recall_curve` draws these as vertical steps. A "zigzag" or locally upward-moving PR curve is a sign of a good model that has discovered a tight cluster of fraud at a specific score level.

</details>

## Next Steps

- **Notebook 9:** Cost-sensitive evaluation — when missing fraud costs \$500 but a false alarm costs \$5
- **Notebook 10:** Calibration curves — are my predicted probabilities actually well-calibrated?
- **Notebook 11:** Cross-validation strategies for imbalanced data (stratified k-fold, repeated CV)

---
*Week 7 — Model Evaluation, Validation & Metrics*